Download all the requirements

In [ ]:
%pip install torch==2.7.1 torchvision torchaudio==2.7.1+cu126 --index-url https://download.pytorch.org/whl/cu126

In [ ]:
%pip install torchcodec ffmpeg-python pandas tqdm scikit-learn librosa birdnetlib birdnet resampy wandb

In [ ]:
%pip freeze > requirements.txt

In [ ]:
# Download dataset
import kagglehub

# Download latest version
kagglehub.auth.set_kaggle_api_token('KGAT_fb4d6921c565524358c1914efc082ed4')
path = kagglehub.competition_download('birdclef-2026', output_dir=os.path.join("birdclef-2026"))

print("Path to competition files:", path)

Dataset and CNN

In [ ]:
# DataSet & DataLoader
import os, ast, torch, torchaudio
import pandas as pd
from torch.utils.data import Dataset

class BirbSet(Dataset):
    # Not gonna pass the sample rate. I trust that the data is formatted at 32k as the competition says.
    def __init__(self, df, root, clip_length, label_to_idx, is_train=False):
        # Needed for the sake of the dataset itself
        self.clips            = []
        self.start_times      = []
        self.end_times        = []
        # Info from the csv file
        self.labels           = []
        self.secondary_labels = []
        self.ratings          = [] # Consider using this field somehow

        self.clip_length      = clip_length   # How long we want each chunk to be. Default to 5 seconds for competition standard
        self.sample_rate      = 32000         # Carried from the competition data description
        self.label_to_idx     = label_to_idx
        self.is_train         = is_train
        self.root             = root

        # First Augmentation: SpecAugment! Uncomment later
        # Spectrogram transforms
        self.amp_to_db = torchaudio.transforms.AmplitudeToDB(stype='power')
        self.mel_spect = torchaudio.transforms.MelSpectrogram(
            sample_rate=self.sample_rate,
            n_fft=800,
            n_mels=64
        )

        # # Augmentation transforms
        # self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=40)
        # self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=16)
        
        # Build manifest arrays
        for _, entry in df.iterrows():

            # The root has to be the train_audio folder in this case
            curr_audio_loc = os.path.join(self.root, os.path.normpath(entry["filename"]))
            # We need to separate each file into blocks of chunk_size
            info = torchaudio.info(curr_audio_loc)
            duration = info.num_frames / self.sample_rate # Trust that sample rate is 32k
            
            pos = 0.0
            # Keep going until we have processed the entire duration
            while pos < duration:
                # If the remaining audio is less than clip_length, cap it at duration
                end_pos = min(pos + self.clip_length, duration)
                
                self.clips.append(curr_audio_loc)
                self.labels.append(self.label_to_idx[entry['primary_label']])
                self.ratings.append(entry.get('rating'))
                self.secondary_labels.append(entry.get('secondary_labels', '[]'))
                
                self.start_times.append(pos)                     
                self.end_times.append(end_pos)     
                
                # Advance by clip_length to check the next segment
                pos += self.clip_length
            

    def __len__(self):
        return len(self.clips)
    
    def __getitem__(self, idx):
        audio_clip = self.clips[idx]
        try:
            frame_offset = int(self.start_times[idx] * self.sample_rate)
            num_frames   = int((self.end_times[idx] - self.start_times[idx]) * self.sample_rate)

            waveform, _ = torchaudio.load(
                audio_clip, frame_offset=frame_offset, num_frames=num_frames
            )

            # Pad or truncate waveform to exact chunk size
            chunk_size  = int(self.sample_rate * self.clip_length) # Wrap in int because the clip_length may be something like 3.2 if it's at the end of the file for example
            current_len = waveform.shape[1]

            if current_len > chunk_size:
                waveform = waveform[:, :chunk_size]
            elif current_len < chunk_size:
                waveform = torch.nn.functional.pad(waveform, (0, chunk_size - current_len))

            # Create Spectrogram
            spectrogram = self.mel_spect(waveform)
            spectrogram = self.amp_to_db(spectrogram)
            
            # Standardize Spectrogram
            mean, std   = spectrogram.mean(), spectrogram.std() + 1e-6
            spectrogram = (spectrogram - mean) / std

            # Initialize target vector
            target = torch.zeros(len(self.label_to_idx), dtype=torch.float32)

            # Set primary label
            primary = self.labels[idx]
            # 1. Safely scale ratings from 1-5 to 0.2-1.0 (so 5.0 rating = 1.0 confidence)
            rating = self.ratings[idx]
            if pd.isna(rating) or rating == 0:
                confidence = 1.0 # Default to full confidence if unrated
            else:
                confidence = rating / 5.0 # A rating of 3.0 becomes 0.6 target

            target[primary] = confidence

            # 2. Apply a much softer penalty for secondary birds (e.g., max 0.3)
            raw_secondary = self.secondary_labels[idx]
            if raw_secondary and raw_secondary not in ('[]', '', None):
                for sec_label in ast.literal_eval(raw_secondary):   
                    if sec_label in self.label_to_idx:
                        target[self.label_to_idx[sec_label]] = confidence * 0.3

            # # FIX: Ensure ALL augmentations are strictly within the is_train block
            # if self.is_train:
            #     spectrogram = self.freq_mask(spectrogram)
            #     spectrogram = self.time_mask(spectrogram)

            #     # Gaussian noise 
            #     if torch.rand(1).item() < 0.5:
            #         spectrogram = spectrogram + torch.randn_like(spectrogram) * 0.1

            #     # Random gain 
            #     gain = torch.empty(1).uniform_(0.75, 1.25)
            #     spectrogram = spectrogram * gain

            return spectrogram, target
            
        except Exception as e:
            print(f"Skipping corrupted/missing file at index {idx} -> {e}. Path: {audio_clip}")
            # FIX: Prevent DataLoader crash by returning an adjacent random index
            return self.__getitem__((idx + 1) % len(self))

In [ ]:
# CNN
import torch.nn as nn
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

class EfficientBirbNN(nn.Module):
    # It is 234 according to the competition description
    def __init__(self, num_classes = 234, pretrained=True):
        super().__init__()
        
        # 1. Load the base EfficientNet model
        weights = EfficientNet_B3_Weights.DEFAULT if pretrained else None
        self.base_model = efficientnet_b3(weights=weights)
        
        # 2. Modify the first convolutional layer to accept 1-channel spectrograms
        # EfficientNet's first layer is located at self.base_model.features[0][0]
        original_conv = self.base_model.features[0][0]
        self.base_model.features[0][0] = nn.Conv2d(
            in_channels=1, 
            out_channels=original_conv.out_channels, 
            kernel_size=original_conv.kernel_size, 
            stride=original_conv.stride, 
            padding=original_conv.padding, 
            bias=False
        )
                
        # 3. Modify the final classification layer for your specific number of bird classes
        in_features = self.base_model.classifier[1].in_features
        self.base_model.classifier[1] = nn.Sequential(
            nn.Dropout(p=0.4), # Extra dropout to prevent overfitting on audio data
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.base_model(x)

In [ ]:
# Instantiate Datasets and DataLoaders
import os
import pandas as pd
from torch.utils.data import DataLoader
# Requires scikit-learn: pip install scikit-learn
from sklearn.model_selection import StratifiedGroupKFold 

# --- Configuration ---
root_path       = os.path.join("..", "birdclef-2026")
CLIP_LENGTH_SEC = 5.0  # Updated to 5.0s to match BirdCLEF 2026 evaluation windows

# 1. Load the master CSV
full_df = pd.read_csv(os.path.join(root_path, "train.csv"))

# 2. Universal label mapping (sorted for reproducibility)
# We can get this from taxonomy.csv
unique_labels = pd.read_csv(os.path.join(root_path, "taxonomy.csv"))
master_label_to_idx = {label: i for i, label in enumerate(unique_labels['primary_label'])}
num_classes         = len(master_label_to_idx)

# 3. Stratified Group Split (Anti-Leakage Validation Scheme)
# This guarantees every class appears in both halves while keeping 
# multiple chunks from the same audio file completely isolated together.
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# We use 'filename' as the grouping key so unique files aren't split across train/val
train_indices, val_indices = next(
    sgkf.split(X=full_df, y=full_df['primary_label'], groups=full_df['filename'])
)

df_train = full_df.iloc[train_indices].reset_index(drop=True)
df_val   = full_df.iloc[val_indices].reset_index(drop=True)

print(f"Train samples: {len(df_train)} | Validation samples: {len(df_val)}")

# 5. DataLoaders
dset_train = BirbSet(
    df=df_train, 
    root=os.path.join(root_path, 'train_audio'), 
    clip_length=CLIP_LENGTH_SEC,
    label_to_idx=master_label_to_idx, 
    is_train=True  # Enables frequency/time masking & audio augmentations
)
loader = DataLoader(dset_train, batch_size=32, shuffle=True, pin_memory=False, num_workers=4)

dset_val = BirbSet(
    df=df_val, 
    root=os.path.join(root_path, 'train_audio'), 
    clip_length=CLIP_LENGTH_SEC,
    label_to_idx=master_label_to_idx, 
    is_train=False  # Keeps validation data pristine and predictable
)
loader_val = DataLoader(dset_val, batch_size=32, shuffle=False, pin_memory=False, num_workers=4)

Training and validation

In [ ]:
# Hyper params
import torch
device = torch.device("cuda")
model = EfficientBirbNN().to(device)
optimiser = torch.optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)

In [ ]:
# Logging
import wandb

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="pumpkin_person-tu-dresden",
    # Set the wandb project where this run will be logged.
    project="birbs-team-project",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 0.0001,
        "architecture": "CNN",
        "dataset": "BirdClef+ 2026",
        "epochs": 30,
    },
)

In [ ]:
from torch.amp import autocast, GradScaler
from tqdm.notebook import tqdm
import numpy as np

# 1. Swapped roc_auc_score for average_precision_score
from sklearn.metrics import average_precision_score

# Loss & Scaler setup
criterion = nn.BCEWithLogitsLoss()
scaler = GradScaler('cuda')

def train_epoch(model, dataloader, optimizer, epoch):
    model.train()
    total_loss = 0.0
    
    # Check if dataloader is empty/hanging right away
    if len(dataloader) == 0:
        print("Warning: DataLoader has 0 batches. Check your dataset.")
        return 0.0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Train]", dynamic_ncols=True)
    
    for spectrograms, targets in pbar:
        # 1. Ensure shapes and types match exactly for BCEWithLogitsLoss
        spectrograms = spectrograms.to('cuda', non_blocking=True)
        targets = targets.to('cuda', dtype=torch.float32, non_blocking=True)
        
        # If targets are 1D (batch_size) and logits are 2D (batch_size, classes)
        if targets.ndim == 1:
            targets = targets.unsqueeze(1) 
        
        optimizer.zero_grad(set_to_none=True) # Slightly faster than zero_grad()
        
        # Mixed precision
        with autocast('cuda'):
            logits = model(spectrograms)
            loss = criterion(logits, targets)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        loss_val = loss.item()
        total_loss += loss_val
        pbar.set_postfix({'loss': f"{loss_val:.4f}"})
        
    return total_loss / len(dataloader)


@torch.no_grad()
def validate_epoch(model, dataloader, epoch):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Val]")
    
    for spectrograms, targets in pbar:
        spectrograms = spectrograms.to('cuda', non_blocking=True)
        targets = targets.to('cuda', dtype=torch.float32, non_blocking=True)
        
        if targets.ndim == 1:
            targets = targets.unsqueeze(1)
            
        with autocast('cuda'):
            logits = model(spectrograms)
            loss = criterion(logits, targets)
            
        total_loss += loss.item()
        
        # Apply sigmoid before pushing to CPU to keep calculations on GPU
        probs = torch.sigmoid(logits).cpu().numpy()
        all_preds.append(probs)
        all_targets.append(targets.cpu().numpy())
        
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    # Concatenate all batches
    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)
    
    # 2. Metric Safeguard Update for mAP
    binary_targets = (all_targets > 0.5).astype(int)
    
    # mAP only requires at least one positive sample to calculate Recall.
    # It does NOT need negative samples like ROC-AUC does.
    valid_classes = np.any(binary_targets == 1, axis=0)
    
    if not np.any(valid_classes):
        print("Warning: No valid classes found for mAP calculation in this split.")
        val_map = 0.0
    else:
        try:
            val_map = average_precision_score(
                binary_targets[:, valid_classes], 
                all_preds[:, valid_classes], 
                average='macro'
            )
        except ValueError as e:
            print(f"mAP calculation error: {e}")
            val_map = 0.0
            
    return total_loss / len(dataloader), val_map

In [ ]:
# 3. Main Loop with Early Stopping & Checkpointing
# ==========================================
MAX_EPOCHS = 30
PATIENCE = 12  # Good baseline for highly imbalanced 234-class datasets
patience_counter = 0
best_val_map = -1.0

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = train_epoch(model, loader, optimiser, epoch)
    val_loss, val_map = validate_epoch(model, loader_val, epoch)

    print(f"Epoch {epoch} Summary -> Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val mAP: {val_map:.4f}")
    run.log({"Training Loss": train_loss, "Val Loss": val_loss, "Val mAP": val_map, "Patience": patience_counter/PATIENCE})

    # Check if we have a new best model
    if val_map > best_val_map:
        best_val_map = val_map
        patience_counter = 0  # Reset patience counter
        
        print(f"--> 🔥 New Best Model Saved! (mAP: {best_val_map:.4f})")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimiser.state_dict(),
            'best_val_map': best_val_map,
        }, "best_efficientbirb_model.pth")
    else:
        patience_counter += 1
        print(f"--> No improvement. Early stopping patience: {patience_counter}/{PATIENCE}")

    print("-" * 50)

    # Trigger early stopping
    if patience_counter >= PATIENCE:
        print(f"🛑 Early stopping triggered at Epoch {epoch}. Model has plateaued.")
        break

run.finish()

In [ ]:
import torchaudio
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast

class SoundscapeDataset(Dataset):
    def __init__(self, audio_path, clip_length=5.0, sample_rate=32000):
        self.clip_length = clip_length
        self.sample_rate = sample_rate
        self.chunk_size = int(self.sample_rate * self.clip_length)
        
        # Load the ENTIRE audio file into RAM. 
        # A 10-min file at 32kHz is only ~75MB, so this is perfectly safe and much faster than seeking.
        self.waveform, sr = torchaudio.load(audio_path)
        
        # Failsafe: Resample if the soundscape isn't 32kHz
        if sr != self.sample_rate:
            self.waveform = torchaudio.functional.resample(self.waveform, sr, self.sample_rate)
            
        # Failsafe: Convert to mono if stereo
        if self.waveform.shape[0] > 1:
            self.waveform = self.waveform.mean(dim=0, keepdim=True)
            
        # Drop the trailing audio that doesn't fit perfectly into a 5-sec window
        self.num_chunks = self.waveform.shape[1] // self.chunk_size
        
        # WE MUST USE THE EXACT SAME TRANSFORMS AS TRAINING
        self.amp_to_db = torchaudio.transforms.AmplitudeToDB(stype='power')
        self.mel_spect = torchaudio.transforms.MelSpectrogram(
            sample_rate=self.sample_rate, n_fft=800, n_mels=64
        )

    def __len__(self):
        return self.num_chunks

    def __getitem__(self, idx):
        start_idx = idx * self.chunk_size
        end_idx   = start_idx + self.chunk_size
        
        chunk = self.waveform[:, start_idx:end_idx]
        
        # Generate and Standardize Spectrogram
        spectrogram = self.mel_spect(chunk)
        spectrogram = self.amp_to_db(spectrogram)
        mean, std   = spectrogram.mean(), spectrogram.std() + 1e-6
        spectrogram = (spectrogram - mean) / std
        
        # Calculate the end time of this chunk (5, 10, 15... etc.)
        end_time = (idx + 1) * int(self.clip_length)
        
        return spectrogram, end_time

In [ ]:
import os
import glob
import torch
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader
from torch.amp import autocast
from tqdm import tqdm

@torch.no_grad()
def generate_soundscape_predictions(model, soundscapes_dir, label_to_idx, device='cuda'):
    """
    Processes a full folder of soundscapes and generates a Kaggle-formatted submission DataFrame.
    """
    model.eval()
    
    # Get all audio files in the folder
    audio_files = glob.glob(os.path.join(soundscapes_dir, "*.ogg"))
    if not audio_files:
        raise FileNotFoundError(f"No .ogg files found in {soundscapes_dir}")
        
    print(f"Found {len(audio_files)} soundscapes. Running inference...")
    
    # Create the column names exactly as Kaggle expects (sorted alphabetically)
    # The reverse mapping helps us build the columns correctly
    idx_to_label = {v: k for k, v in label_to_idx.items()}
    sorted_labels = [idx_to_label[i] for i in range(len(idx_to_label))]
    
    all_rows = []
    
    for audio_path in tqdm(audio_files, desc="Processing Soundscapes"):
        # Extract the soundscape ID (e.g., "10534_COR" from "10534_COR.ogg")
        file_id = os.path.basename(audio_path).split('.')[0]
        
        # Re-use the SoundscapeDataset from the previous step
        dataset = SoundscapeDataset(audio_path, clip_length=5.0)
        loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=0)
        
        for spectrograms, end_times in loader:
            spectrograms = spectrograms.to(device, non_blocking=True)
            
            with autocast('cuda'):
                logits = model(spectrograms)
                probs = torch.sigmoid(logits).cpu().numpy()
            
            end_times = end_times.numpy()
            
            # Build the rows for this batch
            for i in range(len(probs)):
                row_id = f"{file_id}_{end_times[i]}"
                
                # Create a dictionary for this row: {'row_id': '...', 'acafly': 0.01, 'acowoo': 0.05, ...}
                row_data = {'row_id': row_id}
                for class_idx, prob in enumerate(probs[i]):
                    row_data[idx_to_label[class_idx]] = prob
                    
                all_rows.append(row_data)
                
    # Compile into a DataFrame
    submission_df = pd.DataFrame(all_rows)
    
    # Ensure columns are ordered: row_id, then alphabetical bird labels
    cols = ['row_id'] + sorted_labels
    submission_df = submission_df[cols]
    
    return submission_df

In [ ]:
from sklearn.metrics import roc_auc_score

def time_to_seconds(time_str):
    """Converts a timestamp like '00:00:05' into integer seconds (5)."""
    h, m, s = map(int, str(time_str).split(':'))
    return h * 3600 + m * 60 + s

def evaluate_submission(submission_df, ground_truth_path):
    """
    Parses the raw Kaggle soundscape labels and compares them against predictions.
    """
    print("\nLoading and parsing ground truth labels...")
    raw_gt_df = pd.read_csv(ground_truth_path)
    
    # 1. Reconstruct the row_id to match the predictions (e.g., "filename_5")
    file_ids = raw_gt_df['filename'].str.replace('.ogg', '', regex=False)
    end_secs = raw_gt_df['end'].apply(time_to_seconds)
    raw_gt_df['row_id'] = file_ids + '_' + end_secs.astype(str)
    
    # 2. Identify the bird classes we are predicting
    bird_columns = [c for c in submission_df.columns if c != 'row_id']
    
    # 3. Build a one-hot encoded matrix for the ground truth
    # Start with all zeros
    gt_matrix = pd.DataFrame(0, index=range(len(raw_gt_df)), columns=bird_columns)
    gt_matrix['row_id'] = raw_gt_df['row_id'].values
    
    # Populate the 1s for the birds actually present
    for idx, row in raw_gt_df.iterrows():
        # Handle cases where there might be 'nocall' or NaN
        labels_str = str(row['primary_label'])
        if labels_str.lower() != 'nocall' and labels_str != 'nan':
            # Split the semicolon-separated IDs
            active_birds = labels_str.split(';')
            for bird in active_birds:
                bird = bird.strip()
                if bird in bird_columns:
                    gt_matrix.at[idx, bird] = 1.0
                
    # 4. Align the dataframes (critical if any rows are missing or out of order)
    print("Aligning predictions with ground truth...")
    merged = pd.merge(gt_matrix, submission_df, on='row_id', suffixes=('_true', '_pred'))
    
    if len(merged) == 0:
        return "Error: Could not match any row_ids between predictions and ground truth."
    
    # 5. Extract aligned true and predicted matrices
    y_true = merged[[f"{c}_true" for c in bird_columns]].values
    y_pred = merged[[f"{c}_pred" for c in bird_columns]].values
    
    # 6. Metric Safeguard: Only score classes that actually appear in this evaluation set
    # ROC-AUC requires at least one positive (1) and one negative (0) sample per class.
    valid_classes = np.any(y_true == 1, axis=0) & np.any(y_true == 0, axis=0)
    
    if not np.any(valid_classes):
        return "Error: No valid classes in ground truth to evaluate (no positive samples)."
        
    print(f"Scoring {valid_classes.sum()} valid classes...")
    
    # 7. Calculate Macro ROC-AUC
    macro_auc = roc_auc_score(
        y_true[:, valid_classes], 
        y_pred[:, valid_classes], 
        average='macro'
    )
    
    return macro_auc

In [ ]:
# --- Configuration ---
SOUNDSCAPES_DIR = os.path.join(root_path, "train_soundscapes")
LABELS_CSV      = os.path.join(root_path, "train_soundscapes_labels.csv")

if __name__ == "__main__":
    # 1. Load the best saved model from training
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = EfficientBirbNN(num_classes=num_classes)
    
    # Load the weights we saved in the early stopping block earlier
    checkpoint = torch.load("best_efficientbirb_model.pth")
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    
    # 2. Run inference on the whole folder
    print("Starting soundscape inference...")
    submission_df = generate_soundscape_predictions(
        model=model, 
        soundscapes_dir=SOUNDSCAPES_DIR, 
        label_to_idx=master_label_to_idx, 
        device=device
    )
    
    # 3. Save it exactly as Kaggle expects
    submission_df.to_csv("submission.csv", index=False)
    print("Saved submission.csv")
    
    # 4. Evaluate against the ground truth
    if os.path.exists(LABELS_CSV):
        final_score = evaluate_submission(submission_df, LABELS_CSV)
        print(f"\n{'='*40}")
        print(f"🔥 FINAL SOUNDSCAPE MACRO ROC-AUC: {final_score:.4f}")
        print(f"{'='*40}")
    else:
        print(f"Could not find ground truth file at {LABELS_CSV}. Evaluation skipped.")